# House Prices - Data Preparation

This notebook prepares the House Prices dataset (Ames, Iowa housing data) for machine learning.
Steps: exploratory analysis, missing value handling, type fixing, categorical encoding, and feature/target split.

**Dataset:** [House Prices - Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)

## 1. Load and Explore the Data

Load the dataset and check its shape, structure, and missing values to understand what cleaning is needed.

In [2]:
import pandas as pd

df = pd.read_csv('train.csv')

print(df.shape)
print(df.head())
print(df.info())
print(df.isnull().sum().sort_values(ascending=False).head(20))

(1460, 81)
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  
0   2008  

In [ ]:
## 2. Handle Missing Values

Referring to `data_description.txt`, most "missing" values in this dataset aren't actually missing —
they indicate the house doesn't have that feature (e.g., `NaN` in `PoolQC` means "No Pool").

- Columns like `PoolQC`, `Alley`, `Fence`, `FireplaceQu`, garage/basement fields → filled with `'None'`
- `GarageYrBlt`, `MasVnrArea` → filled with `0` (no garage/veneer means no year/area)
- `LotFrontage` → filled with the **median grouped by Neighborhood**, since lot sizes vary a lot by area
- `Electrical` → only 1 missing value, filled with the **mode** (most common value)

In [3]:
# Columns where NaN means "feature doesn't exist" -> fill with "None"
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'MasVnrType']

for col in none_cols:
    df[col] = df[col].fillna('None')

# GarageYrBlt: no garage means no year -> fill with 0
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(0)

# MasVnrArea: no veneer means 0 area
df['MasVnrArea'] = df['MasVnrArea'].fillna(0)

# LotFrontage: genuinely missing numeric value -> fill with median (grouped by neighborhood is even better)
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

# Electrical: only 1 missing -> fill with the most common value (mode)
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

# Confirm no missing values remain
print(df.isnull().sum().sum())

0


## 3. Encode Ordinal Categorical Variables

Several columns represent quality/condition on a natural scale (Poor → Excellent), so they're mapped to
numbers that preserve that order rather than one-hot encoded. This keeps the "better than" relationship
intact for the model (e.g., Excellent > Good > Average).

Columns like `BsmtExposure`, `BsmtFinType1/2`, `GarageFinish`, `Functional`, `Fence`, `LandSlope`,
and `PavedDrive` have their own specific ordinal scales, mapped individually based on `data_description.txt`.

In [5]:
# Quality/condition columns using Po-Fa-TA-Gd-Ex scale (plus 'None' for missing feature)
quality_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
                 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

for col in quality_cols:
    df[col] = df[col].map(quality_map)

print(df[quality_cols].head())

   ExterQual  ExterCond  BsmtQual  BsmtCond  HeatingQC  KitchenQual  \
0          4          3         4         3          5            4   
1          3          3         4         3          5            3   
2          4          3         4         3          5            4   
3          3          3         3         4          4            4   
4          4          3         4         3          5            4   

   FireplaceQu  GarageQual  GarageCond  PoolQC  
0            0           3           3       0  
1            3           3           3       0  
2            3           3           3       0  
3            4           3           3       0  
4            3           3           3       0  


In [6]:
# BsmtExposure: basement exposure to outside
df['BsmtExposure'] = df['BsmtExposure'].map({'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4})

# BsmtFinType1 & BsmtFinType2: basement finish quality
fintype_map = {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
df['BsmtFinType1'] = df['BsmtFinType1'].map(fintype_map)
df['BsmtFinType2'] = df['BsmtFinType2'].map(fintype_map)

# GarageFinish: how finished the garage interior is
df['GarageFinish'] = df['GarageFinish'].map({'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3})

# Functional: home functionality rating (Typ = typical/best)
df['Functional'] = df['Functional'].map({'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3,
                                            'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7})

# Fence: fence quality
df['Fence'] = df['Fence'].map({'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4})

# LandSlope: gentle to severe slope
df['LandSlope'] = df['LandSlope'].map({'Gtl': 0, 'Mod': 1, 'Sev': 2})

# PavedDrive
df['PavedDrive'] = df['PavedDrive'].map({'N': 0, 'P': 1, 'Y': 2})

# Check for any NaN introduced by mapping (would mean an unexpected category slipped through)
ordinal_cols = ['BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageFinish',
                 'Functional', 'Fence', 'LandSlope', 'PavedDrive']
print(df[ordinal_cols].isnull().sum())

BsmtExposure    0
BsmtFinType1    0
BsmtFinType2    0
GarageFinish    0
Functional      0
Fence           0
LandSlope       0
PavedDrive      0
dtype: int64


## 4. Encode Nominal Categorical Variables

Remaining categorical columns (like `Neighborhood`, `HouseStyle`, `RoofStyle`) have no natural order,
so they're one-hot encoded — each category becomes its own 0/1 column.
`drop_first=True` avoids redundant columns and multicollinearity.

In [7]:
remaining_categorical = df.select_dtypes(include='str').columns.tolist()
print(remaining_categorical)
print(len(remaining_categorical))

['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'GarageType', 'MiscFeature', 'SaleType', 'SaleCondition']
25


In [8]:
df = pd.get_dummies(df, columns=remaining_categorical, drop_first=True)

print(df.shape)
print(df.select_dtypes(include='str').columns.tolist())  # should be empty now

(1460, 206)
[]


## 5. Split Features and Target

Separate the dataset into `X` (features) and `y` (`SalePrice`, the target we'd predict in a modeling step),
and confirm zero missing values remain.

In [9]:
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

print(X.shape)
print(y.shape)
print(X.isnull().sum().sum())  # final check - should be 0

(1460, 205)
(1460,)
0


## Summary

- Started with 1,460 rows × 81 columns, several with missing/messy values
- All missing values handled based on documented column meanings, not blindly dropped
- All categorical columns encoded (ordinal where order matters, one-hot otherwise)
- Final dataset: 1,460 rows × 205 features, ready for model training